# Python Multithreading and Multiprocessing — Complete Practical Guide

A practical notebook covering concurrency, parallelism, threads, processes, synchronization, executors, inter-process communication, performance and real-world architecture.

## Learning objectives
- Distinguish concurrency, parallelism, threads and processes.
- Choose a model for CPU-bound and I/O-bound workloads.
- Understand the CPython Global Interpreter Lock (GIL).
- Use `threading`, synchronization primitives and thread-safe queues.
- Use `ThreadPoolExecutor` and `ProcessPoolExecutor`.
- Share data safely between processes.
- Avoid race conditions, deadlocks and common multiprocessing errors.
- Benchmark different execution strategies.

**Important for Windows/Jupyter:** Thread examples run directly in notebooks. Multiprocessing examples below are script-ready. For reliable execution on Windows, copy each complete multiprocessing example into a `.py` file, define worker functions at module level, and use `if __name__ == "__main__":`. This is necessary because Windows commonly uses the `spawn` start method.

## 1. Concurrency and parallelism

**Concurrency** means multiple tasks can make progress during overlapping periods. **Parallelism** means multiple tasks execute at the same instant, typically on separate CPU cores.

Concurrency is about structuring work; parallelism is about simultaneous execution. A program can be concurrent without being parallel.

Examples:
- A web service handles several requests while each waits for network responses.
- A data-processing program calculates several independent batches on different CPU cores.

## 2. Process vs thread

| Feature | Thread | Process |
|---|---|---|
| Unit | Execution path inside a process | Independent running program |
| Memory | Shares process memory | Separate address space by default |
| Communication | Shared objects, queues, locks | Queues, pipes, shared memory |
| Overhead | Usually lower | Usually higher |
| Isolation | Less isolated | More isolated |
| Common use | I/O-bound work | CPU-bound work |

A process can contain multiple threads. Threads share process resources but have their own execution stacks.

## 3. CPU-bound vs I/O-bound

**CPU-bound:** Most time is spent calculating: image transformations, pure-Python numerical loops, compression or expensive parsing.

**I/O-bound:** Most time is spent waiting: API calls, file reads, database queries and downloads.

Starting point:
- I/O-bound: threads or asynchronous I/O.
- CPU-bound pure Python: processes.
- Native numerical libraries: benchmark first; some release the GIL or use their own parallelism.

## 4. The Global Interpreter Lock (GIL)

In a conventional CPython build, the GIL allows only one thread at a time to execute Python bytecode within an interpreter. This means CPU-heavy pure-Python threads usually do not achieve multi-core parallelism.

Threads are still useful for I/O because a thread waiting on I/O can let another thread run. Multiple processes have separate interpreters and GILs, so CPU-bound work can execute across cores.

The GIL does not make application-level shared data safe. Race conditions are still possible. Some Python builds support free-threaded execution, so check the actual Python build and benchmark rather than assuming all installations behave identically.

## 5. First thread with `threading.Thread`

In [ ]:
import threading
import time

def task(task_name, delay):
    print(f"{task_name}: started")
    time.sleep(delay)  # Simulates I/O waiting
    print(f"{task_name}: completed")

thread_1 = threading.Thread(target=task, args=("Task-A", 2))
thread_2 = threading.Thread(target=task, args=("Task-B", 1))

thread_1.start()
thread_2.start()

thread_1.join()
thread_2.join()

print("Main thread: all tasks finished")

Key methods:
- `Thread(target=..., args=...)`: create a thread object.
- `start()`: start execution in a new thread.
- `join()`: wait for completion.
- `is_alive()`: check whether it is running.
- `name`: inspect or set the thread name.
- `daemon`: configure daemon behavior.

Calling `run()` directly does not start a new thread; use `start()`.

## 6. Overlapping I/O-bound tasks

In [ ]:
import threading
import time

def download_simulation(file_name, delay):
    print("Starting", file_name)
    time.sleep(delay)
    print("Finished", file_name)

files = [("file_A", 2), ("file_B", 1), ("file_C", 3)]
started = time.perf_counter()

threads = []
for name, delay in files:
    thread = threading.Thread(
        target=download_simulation, args=(name, delay)
    )
    threads.append(thread)
    thread.start()

for thread in threads:
    thread.join()

print(f"Elapsed: {time.perf_counter() - started:.2f} seconds")

Because the tasks mostly wait, their durations overlap. Total time should be near the longest wait plus overhead, rather than the sum of all waits. Real network and disk workloads have their own resource limits.

## 7. Race conditions — why shared state is dangerous

In [ ]:
import threading

counter = 0

def unsafe_increment():
    global counter
    for _ in range(100_000):
        current = counter
        current += 1
        counter = current

threads = [
    threading.Thread(target=unsafe_increment)
    for _ in range(4)
]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()

print("Counter:", counter)
print("Expected:", 400_000)

A race condition occurs when a result depends on the timing/interleaving of operations. The read-modify-write sequence is a compound operation. The observed result may vary and the bug may not reproduce on every run.

## 8. `Lock` — protect critical sections

In [ ]:
import threading

counter = 0
counter_lock = threading.Lock()

def safe_increment():
    global counter
    for _ in range(100_000):
        with counter_lock:
            counter += 1

threads = [
    threading.Thread(target=safe_increment)
    for _ in range(4)
]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()

print("Counter:", counter)
print("Expected:", 400_000)

Prefer `with lock:` so the lock is released even if an exception occurs. Keep critical sections short. Avoid holding a lock during slow network or disk operations unless correctness requires it.

## 9. `RLock` — re-entrant lock

In [ ]:
import threading

class Account:
    def __init__(self, balance):
        self.balance = balance
        self.lock = threading.RLock()

    def deposit(self, amount):
        with self.lock:
            if amount <= 0:
                raise ValueError("Amount must be positive")
            self.balance += amount

    def deposit_with_audit(self, amount):
        with self.lock:
            self._record_audit()
            self.deposit(amount)  # Acquires the same RLock again

    def _record_audit(self):
        print("Audit event recorded")

account = Account(1000)
account.deposit_with_audit(250)
print(account.balance)

A regular `Lock` cannot be acquired again by its owning thread before release. `RLock` allows re-entry but must be released the same number of times it was acquired. Use it only when re-entrant locking is genuinely needed.

## 10. `Semaphore` — limit concurrent access

In [ ]:
import threading
import time

api_slots = threading.Semaphore(2)

def call_service(request_id):
    with api_slots:
        print("Entered service:", request_id)
        time.sleep(1)
        print("Left service:", request_id)

threads = [
    threading.Thread(target=call_service, args=(i,))
    for i in range(1, 7)
]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()

At most two threads can enter the protected section at once. Semaphores can limit concurrent API calls, database connections or access to a constrained resource. `BoundedSemaphore` can help detect over-release mistakes.

## 11. `Event` — signal between threads

In [ ]:
import threading
import time

data_ready = threading.Event()

def consumer():
    print("Consumer waiting")
    data_ready.wait()
    print("Consumer: data is ready")

def producer():
    print("Producer preparing data")
    time.sleep(2)
    data_ready.set()

t1 = threading.Thread(target=consumer)
t2 = threading.Thread(target=producer)
t1.start()
t2.start()
t1.join()
t2.join()

An event is a shared flag. `wait(timeout=...)` can wait with a timeout; `set()` signals; `clear()` resets. An event signals that something happened but does not carry a data payload.

## 12. `Condition` — wait for a state change

In [ ]:
import threading
import time

condition = threading.Condition()
items = []

def consumer():
    with condition:
        while not items:
            print("Waiting for an item")
            condition.wait()
        item = items.pop(0)
        print("Received:", item)

def producer():
    time.sleep(1)
    with condition:
        items.append("new-data")
        condition.notify()

t1 = threading.Thread(target=consumer)
t2 = threading.Thread(target=producer)
t1.start()
t2.start()
t1.join()
t2.join()

Always check the condition in a `while` loop. A thread can wake before the condition is satisfied. `wait()` releases the underlying lock while waiting and reacquires it before returning. `notify()` wakes one waiter; `notify_all()` wakes all waiters.

## 13. Producer-consumer with `queue.Queue`

In [ ]:
import queue
import threading
import time

work_queue = queue.Queue()

def producer():
    for task_id in range(1, 6):
        work_queue.put(task_id)
    work_queue.put(None)  # Sentinel: one consumer in this example

def consumer():
    while True:
        item = work_queue.get()
        try:
            if item is None:
                return
            print("Processing:", item)
            time.sleep(0.2)
        finally:
            work_queue.task_done()

producer_thread = threading.Thread(target=producer)
consumer_thread = threading.Thread(target=consumer)

producer_thread.start()
consumer_thread.start()

producer_thread.join()
work_queue.join()
consumer_thread.join()
print("Queue processing completed")

`queue.Queue` handles thread synchronization. Call `task_done()` once for every retrieved item if using `join()`. A bounded queue (`queue.Queue(maxsize=...)`) applies backpressure when producers are faster than consumers. With multiple consumers, send one sentinel per consumer.

## 14. Daemon and non-daemon threads

In [ ]:
import threading
import time

def background_task():
    for step in range(3):
        print("Background step", step)
        time.sleep(0.4)

worker = threading.Thread(target=background_task, daemon=False)
worker.start()
worker.join()
print("Main exits after worker")

A non-daemon thread keeps the process alive until it finishes. A daemon thread does not prevent interpreter shutdown and may be stopped abruptly. For important work, use non-daemon threads and an explicit shutdown signal such as `Event`.

## 15. Thread pools with `ThreadPoolExecutor`

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

def fetch_record(record_id):
    time.sleep(0.5)  # Simulate I/O
    return {"record_id": record_id, "status": "loaded"}

with ThreadPoolExecutor(max_workers=3) as executor:
    futures = [
        executor.submit(fetch_record, record_id)
        for record_id in range(1, 7)
    ]

    for future in as_completed(futures):
        print(future.result())

Useful executor APIs:
- `submit(fn, *args)`: schedule one call and return a `Future`.
- `map(fn, iterable)`: yield results in input order.
- `as_completed(futures)`: iterate futures as they finish.
- `future.result()`: return result or re-raise worker exception.
- `future.exception()`: inspect a task exception.
- `future.cancel()`: cancel work that has not started.

The executor context manager waits for submitted work to finish.

## 16. `map()` vs `submit()` vs `as_completed()`

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

def task(number):
    time.sleep(0.1 * (4 - number))
    return number * number

with ThreadPoolExecutor(max_workers=4) as executor:
    print("map: input order")
    print(list(executor.map(task, [1, 2, 3, 4])))

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [executor.submit(task, n) for n in [1, 2, 3, 4]]
    print("as_completed: completion order")
    for future in as_completed(futures):
        print(future.result())

## 17. Multiprocessing fundamentals

The `multiprocessing` module creates independent processes. Each process has its own memory space and Python interpreter.

On Windows, `spawn` starts a fresh interpreter and imports the main module. Therefore:
- Worker functions must be defined at module level.
- Arguments and results must be picklable.
- Process creation must be protected by `if __name__ == "__main__":`.
- Do not create pools as an import-time side effect.
- Run examples in `.py` files if notebook process startup fails.

## 18. First process pool — `ProcessPoolExecutor`

Copy this complete example into `process_square_demo.py` and run `python process_square_demo.py`.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import os

def calculate_square(number):
    return {
        "number": number,
        "square": number * number,
        "process_id": os.getpid()
    }

def main():
    with ProcessPoolExecutor(max_workers=4) as executor:
        results = list(executor.map(calculate_square, range(1, 9)))
    for result in results:
        print(result)

if __name__ == "__main__":
    main()

The worker function is at module level so child processes can import it. The main guard prevents child interpreters from recursively creating more pools.

## 19. Managing processes with `multiprocessing.Process`

Copy into `process_demo.py` and run from a terminal.

In [ ]:
from multiprocessing import Process
import os
import time

def worker(name, delay):
    print(f"{name} running in PID={os.getpid()}")
    time.sleep(delay)
    print(f"{name} completed")

def main():
    processes = [
        Process(target=worker, args=("Worker-A", 2)),
        Process(target=worker, args=("Worker-B", 1)),
    ]
    for process in processes:
        process.start()
    for process in processes:
        process.join()
    print("Parent process completed")

if __name__ == "__main__":
    main()

`start()` launches a child process; `join()` waits for it; `exitcode` reports its exit status after completion. A process target's return value is not automatically returned to the parent.

## 20. Benchmark sequential vs threads for CPU work

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

def cpu_work(number):
    total = 0
    for value in range(number):
        total += value * value
    return total

inputs = [800_000] * 4

start = time.perf_counter()
sequential = [cpu_work(n) for n in inputs]
sequential_time = time.perf_counter() - start

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as executor:
    threaded = list(executor.map(cpu_work, inputs))
thread_time = time.perf_counter() - start

print(f"Sequential: {sequential_time:.2f}s")
print(f"Threads:    {thread_time:.2f}s")
print("Equal results:", sequential == threaded)

In conventional CPython, pure-Python CPU work often does not speed up with threads because of the GIL. Thread scheduling can make it slower. Benchmark on your own machine; avoid drawing conclusions from a single run.

## 21. Benchmark sequential vs processes

Copy this script into `cpu_process_benchmark.py` and run it from a terminal.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import time

def cpu_work(number):
    total = 0
    for value in range(number):
        total += value * value
    return total

def main():
    inputs = [800_000] * 4

    start = time.perf_counter()
    sequential = [cpu_work(n) for n in inputs]
    sequential_time = time.perf_counter() - start

    start = time.perf_counter()
    with ProcessPoolExecutor(max_workers=4) as executor:
        parallel = list(executor.map(cpu_work, inputs))
    process_time = time.perf_counter() - start

    print(f"Sequential: {sequential_time:.2f}s")
    print(f"Processes:  {process_time:.2f}s")
    print("Equal results:", sequential == parallel)

if __name__ == "__main__":
    main()

Processes can use multiple cores, but startup, serialization and inter-process communication add overhead. For small tasks, sequential execution may be faster.

## 22. Process communication with `multiprocessing.Queue`

Processes do not share ordinary Python variables. A multiprocessing queue transfers picklable objects.

In [ ]:
# Save as process_queue_demo.py and run from a terminal.
from multiprocessing import Process, Queue

def producer(output_queue):
    for number in range(1, 6):
        output_queue.put(number * number)
    output_queue.put(None)

def consumer(input_queue):
    while True:
        item = input_queue.get()
        if item is None:
            break
        print("Received:", item)

def main():
    q = Queue()
    p1 = Process(target=producer, args=(q,))
    p2 = Process(target=consumer, args=(q,))
    p1.start()
    p2.start()
    p1.join()
    p2.join()

if __name__ == "__main__":
    main()

A multiprocessing queue serializes objects for transfer. The receiver gets a separate object representation. Use sentinels or another explicit shutdown protocol so consumers know when to stop.

## 23. Shared numeric state with `multiprocessing.Value`

Shared memory needs synchronization when multiple processes update the same value.

In [ ]:
# Save as shared_value_demo.py and run from a terminal.
from multiprocessing import Process, Value, Lock

def increment(counter, lock):
    for _ in range(10_000):
        with lock:
            counter.value += 1

def main():
    counter = Value("i", 0)
    lock = Lock()
    processes = [
        Process(target=increment, args=(counter, lock))
        for _ in range(4)
    ]
    for p in processes:
        p.start()
    for p in processes:
        p.join()
    print("Counter:", counter.value)
    print("Expected:", 40_000)

if __name__ == "__main__":
    main()

`Value` and `Array` suit simple shared numeric data. For complex application state, queue-based message passing is often easier to reason about than shared mutable objects.

## 24. Shared memory for larger arrays

`multiprocessing.shared_memory` exposes a memory block to multiple processes. This example uses NumPy and must be run as a script.

In [ ]:
# Save as shared_memory_demo.py. Requires NumPy.
from multiprocessing import Process
from multiprocessing.shared_memory import SharedMemory
import numpy as np

def add_one(shm_name, shape, dtype):
    existing = SharedMemory(name=shm_name)
    try:
        array = np.ndarray(shape, dtype=dtype, buffer=existing.buf)
        array += 1
    finally:
        existing.close()

def main():
    original = np.arange(10, dtype=np.int64)
    shm = SharedMemory(create=True, size=original.nbytes)
    try:
        shared = np.ndarray(original.shape, dtype=original.dtype, buffer=shm.buf)
        shared[:] = original
        p = Process(target=add_one, args=(shm.name, original.shape, original.dtype))
        p.start()
        p.join()
        print(shared)
    finally:
        shm.close()
        shm.unlink()

if __name__ == "__main__":
    main()

The creator should `unlink()` the block when it is no longer needed. Each process closes its own handle. Shared memory does not automatically provide synchronization; overlapping writes still need coordination.

## 25. Multiprocessing start methods

- **spawn:** Starts a fresh interpreter and imports the main module. Common on Windows and macOS.
- **fork:** Starts by forking the current process. Available on many Unix-like systems, but can be unsafe after threads or complex native libraries have started.
- **forkserver:** A server process forks children on request. Available on supported Unix-like systems.

Inspect the current runtime:

```python
import multiprocessing as mp
print(mp.get_all_start_methods())
print(mp.get_start_method())
```

For portable code, avoid relying on implicit platform defaults. If selecting a context, use `multiprocessing.get_context("spawn")` and create related primitives from that context.

## 26. Process futures and exception handling

Copy into `process_futures_demo.py` and run from a terminal.

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed

def divide_task(pair):
    numerator, denominator = pair
    if denominator == 0:
        raise ZeroDivisionError("Denominator cannot be zero")
    return numerator / denominator

def main():
    inputs = [(10, 2), (20, 4), (5, 0), (30, 5)]
    with ProcessPoolExecutor(max_workers=3) as executor:
        futures = {executor.submit(divide_task, item): item for item in inputs}
        for future in as_completed(futures):
            item = futures[future]
            try:
                print(item, "=>", future.result())
            except Exception as error:
                print(item, "failed:", repr(error))

if __name__ == "__main__":
    main()

## 27. Choosing a concurrency model

| Situation | Starting point | Why |
|---|---|---|
| Many API requests | Thread pool | Most time waits for I/O |
| Downloading files | Thread pool | Network waits overlap |
| Many independent file reads | Thread pool | I/O-bound, subject to disk limits |
| Pure-Python CPU calculations | Process pool | Separate interpreters can use cores |
| Large NumPy operations | Benchmark | Native code may release GIL/use threads |
| Small quick tasks | Sequential | Pool overhead may dominate |
| Shared mutable state | Minimize it | Synchronization adds complexity |

Measure the actual workload, including process startup, serialization, memory and result-transfer overhead.

## 28. Deadlocks

A deadlock occurs when tasks wait indefinitely for resources held by one another. Example: thread A holds lock 1 and waits for lock 2, while thread B holds lock 2 and waits for lock 1.

Prevention:
1. Acquire multiple locks in a consistent global order.
2. Keep critical sections short.
3. Avoid blocking/unknown calls while holding locks.
4. Use timeouts where appropriate.
5. Prefer queues and message passing over complex shared state.
6. Design shutdown paths before adding workers.
7. Avoid waiting on another task from the same saturated executor if that task needs a free worker.

Example timeout:

```python
acquired = lock.acquire(timeout=2)
if not acquired:
    raise TimeoutError("Could not acquire lock")
try:
    # protected work
    pass
finally:
    lock.release()
```

## 29. Cancellation and graceful shutdown

In [ ]:
import threading
import time

stop_event = threading.Event()

def worker():
    while not stop_event.is_set():
        print("Worker doing one unit of work")
        stop_event.wait(timeout=0.5)
    print("Worker shut down cleanly")

thread = threading.Thread(target=worker)
thread.start()
time.sleep(1.5)
stop_event.set()
thread.join(timeout=2)
print("Still alive:", thread.is_alive())

Threads cannot generally be safely force-killed. Use cooperative cancellation. For executors, `Future.cancel()` cancels work that has not started. Executor shutdown does not forcibly stop tasks already running; running tasks should observe a stop signal.

## 30. Logging in concurrent applications

Python logging is thread-safe for normal use within one process. Multiple processes writing directly to the same rotating file are a different problem: file writes and rotation are not coordinated automatically.

Recommended:
- Threads in one process can use a shared configured logger.
- Multiple processes should send records to a queue/listener or write to platform-collected stdout/stderr.
- Include process ID, thread name and task ID when useful.

In [ ]:
import logging
import threading

logger = logging.getLogger("concurrent_app")
logger.setLevel(logging.INFO)
logger.propagate = False

if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter(
        "%(asctime)s | pid=%(process)d | thread=%(threadName)s | %(levelname)s | %(message)s"
    ))
    logger.addHandler(handler)

def worker(number):
    logger.info("Worker %s started", number)
    logger.info("Worker %s completed", number)

threads = [
    threading.Thread(target=worker, args=(i,), name=f"worker-{i}")
    for i in range(3)
]
for t in threads:
    t.start()
for t in threads:
    t.join()

## 31. Real-world example — concurrent file processing

Use a thread pool for independent file reads. Each worker returns its own result rather than mutating a shared result list.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import logging

logger = logging.getLogger("file_processor")
logger.setLevel(logging.INFO)
logger.propagate = False

if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(levelname)s | %(threadName)s | %(message)s"))
    logger.addHandler(handler)

def process_file(file_path):
    path = Path(file_path)
    logger.info("Reading %s", path.name)
    text = path.read_text(encoding="utf-8")
    return {
        "file": path.name,
        "characters": len(text),
        "lines": len(text.splitlines())
    }

def process_many_files(paths, max_workers=4):
    results, failures = [], []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_path = {
            executor.submit(process_file, path): path for path in paths
        }
        for future in as_completed(future_to_path):
            path = future_to_path[future]
            try:
                results.append(future.result())
            except Exception as error:
                logger.exception("Failed to process %s", path)
                failures.append({"file": str(path), "error": str(error)})
    return results, failures

sample_paths = []
for index in range(3):
    path = Path(f"sample_{index}.txt")
    path.write_text(f"Example file {index}\nSecond line\n", encoding="utf-8")
    sample_paths.append(path)

results, failures = process_many_files(sample_paths)
print(results)
print(failures)

For very large files, stream or process chunks instead of reading everything into memory. More threads do not always mean faster disk reads.

## 32. Real-world example — parallel ML batch inference

A process pool can distribute independent CPU-heavy batches. Model loading can be done once per worker using an initializer. This is an architecture sketch, not a real model integration.

In [ ]:
# Save as parallel_inference.py and run from a terminal.
from concurrent.futures import ProcessPoolExecutor

MODEL = None

def initialize_worker():
    global MODEL
    # Load the actual model here, once in each worker.
    MODEL = "demo-model"

def predict_batch(batch):
    # Replace with MODEL.predict(batch) in a real implementation.
    return [value * value for value in batch]

def main():
    batches = [
        list(range(0, 1000)),
        list(range(1000, 2000)),
        list(range(2000, 3000)),
        list(range(3000, 4000)),
    ]
    with ProcessPoolExecutor(
        max_workers=4,
        initializer=initialize_worker
    ) as executor:
        predictions = list(executor.map(predict_batch, batches))
    print("Batches completed:", len(predictions))
    print("First batch sample:", predictions[0][:5])

if __name__ == "__main__":
    main()

Benchmark model-loading cost, serialization, batch size, worker count, memory and native library thread counts. If a framework already uses multiple CPU threads or a GPU, adding a process pool may reduce performance through oversubscription.

## 33. Common mistakes and fixes

| Mistake | Better approach |
|---|---|
| Threads for every CPU-heavy pure-Python task | Benchmark a process pool |
| Processes for tiny tasks | Include startup/serialization overhead |
| Forget `join()` | Wait when completion matters |
| Call `run()` instead of `start()` | Use `start()` |
| Share mutable state without synchronization | Use locks or message passing |
| Use `queue.Queue` between processes | Use `multiprocessing.Queue` |
| Nested worker function in process pool | Define worker at module level |
| Omit main guard on Windows | Use `if __name__ == "__main__":` |
| Pass unpicklable objects to workers | Pass simple data or redesign setup |
| Multiple processes write one rotating file | Centralize logging |
| Assume daemon threads finish cleanup | Use cooperative shutdown |
| Create excessive workers | Benchmark and respect resource limits |

## 34. Decision guide

1. Mostly waiting for network, file or database I/O? Start with `ThreadPoolExecutor`; for very high I/O concurrency, evaluate `asyncio`.
2. Mostly pure-Python CPU computation? Start with `ProcessPoolExecutor`.
3. Small, quick tasks? Start sequentially; pool overhead may dominate.
4. Need shared state? Prefer immutable inputs and returned results. If shared access is necessary, use locks, queues or shared memory deliberately.
5. Heavy native library work? Benchmark its own threading behavior before adding another concurrency layer.

## 35. Advanced practice — implement without solutions

Each problem is a realistic implementation challenge. Every prompt is followed by a blank code cell.

### Problem 1 — Concurrent API Data Collector

Build a service that collects data from many independent endpoints.

**Requirements**
- Use `ThreadPoolExecutor` with a configurable worker limit.
- Simulate or call independent endpoints.
- Add request timeout and retry handling for temporary failures.
- Capture result, response time and error status for each endpoint.
- Use `as_completed()` to process finished requests.
- Add structured logging with request IDs.
- Enforce a maximum number of concurrent requests.
- Return successful results separately from failures.

**Acceptance criteria**
- One failing endpoint does not discard successful results.
- Concurrency is bounded.
- Each request can be traced in logs.
- A different endpoint list can be processed without redesign.

### Problem 2 — Thread-Safe Banking Transaction Simulator

Build a multi-threaded banking simulation.

**Requirements**
- Create an `Account` class with ID and balance.
- Support deposit, withdrawal and transfer.
- Protect balances with locks.
- Implement transfers between two accounts without deadlock.
- Validate positive amounts and insufficient funds.
- Run many transactions concurrently.
- Maintain an audit log of completed and rejected transactions.
- Verify that total money remains consistent.

**Acceptance criteria**
- No lost updates or invalid negative balances.
- Transfers are atomic from the application's perspective.
- Lock acquisition follows a consistent order.
- Final balances reconcile with the transaction ledger.

### Problem 3 — Parallel Dataset Processing and Feature Engineering

Build a pipeline that processes independent dataset batches.

**Requirements**
- Divide input data into batches.
- Compare sequential, threaded and process-based implementations.
- Use processes for CPU-heavy pure-Python feature calculations.
- Combine batch results deterministically.
- Handle failed batches without losing successful outputs.
- Record execution time, batch size and worker count.
- Avoid transferring unnecessary large objects.
- Explain when NumPy/pandas may already use native parallelism.

**Acceptance criteria**
- All execution modes produce equivalent outputs.
- Timing results are compared.
- Failures identify the affected batch.
- The chosen strategy is justified by measurements.

### Problem 4 — Producer-Consumer Job Processing System

Build a background job system with multiple producers and consumers.

**Requirements**
- Use `queue.Queue` for thread communication.
- Use a bounded queue for backpressure.
- Run multiple producer and consumer threads.
- Give each job a unique ID and category.
- Implement graceful shutdown with sentinels or an explicit stop protocol.
- Track completed, failed and retried jobs.
- Use locks only for state that actually needs protection.
- Acknowledge every retrieved queue item.

**Acceptance criteria**
- No job is silently lost.
- Consumers shut down cleanly.
- Queue capacity is respected.
- Final job counts reconcile with submitted jobs.

### Problem 5 — Parallel ML Inference Service

Build a process-based inference system for independent data batches.

**Requirements**
- Create a script-ready `ProcessPoolExecutor` implementation.
- Load/initialize the model once per worker.
- Split incoming data into configurable batches.
- Return predictions with batch IDs.
- Handle worker failures and invalid batches.
- Record throughput, duration, worker count and memory considerations.
- Compare sequential and process-based execution.
- Explain whether a process pool fits your ML framework and hardware.

**Acceptance criteria**
- Worker functions are picklable and defined at module level.
- Process creation is protected by the main guard.
- Results retain their batch IDs.
- Conclusions are based on measured results.

## 36. Final revision checklist

- [ ] Explain concurrency vs parallelism.
- [ ] Distinguish threads and processes.
- [ ] Identify CPU-bound and I/O-bound work.
- [ ] Explain the GIL and its practical impact.
- [ ] Create threads and wait with `join()`.
- [ ] Explain race conditions and use `Lock`.
- [ ] Distinguish `Lock`, `RLock`, `Semaphore`, `Event` and `Condition`.
- [ ] Implement producer-consumer with `queue.Queue`.
- [ ] Use `ThreadPoolExecutor`, futures and `as_completed()`.
- [ ] Create processes with `Process` and `ProcessPoolExecutor`.
- [ ] Explain Windows main guard and pickling requirements.
- [ ] Transfer data with multiprocessing queues.
- [ ] Explain shared values and shared memory.
- [ ] Identify deadlocks and implement graceful shutdown.
- [ ] Benchmark sequential, threaded and process-based execution.
- [ ] Choose a model based on workload and measurement.

**Recommended order:** Complete Problems 1 and 2 first. Use Problem 3 to compare execution models on a dataset. Finish with Problems 4 and 5 to build application-level concurrency skills.